# Segmentation mask — candidate rules

**Purpose.** The supervisor said the camera-direction idea is not
worth it, and suggested a rule for the vegetation area
closest to each observation instead; he called it a central point of
the work and said to test several approaches (see
[`docs/meetings/2026-11-04.md`](../../docs/meetings/2026-11-04.md#resposta-dos-orientadores-por-mensagem)).
No rule is chosen here: each candidate is tried on a small sample and
exported to GeoPackage to compare visually in QGIS before deciding.

**Candidates** (none grounded in a paper that does exactly this; see
[`thesis/notes.md`](../../thesis/notes.md#máscara-de-segmentação-testar-várias-abordagens)
for the literature search):
1. **COS 2023 polygon:** if the point falls inside a "Florestas de
   acácias" (`5.1.1.7`) polygon, the mask is that polygon.
2. **Region growing on TESSERA embeddings:** from the pixel's
   embedding, grow to neighbours above a similarity threshold, up to
   a maximum radius.
3. **Superpixels:** segment a window around the point (SLIC) and take
   the superpixel the point falls in.

**Needs** the cleaned file from `02_citizen_science/acacia_observations.ipynb`.

> **Question numbering (note of 2026-11-13).** Current numbering in
> `docs/research_questions.md`: RQ1 = effort-reduction curve (citizen
> science against expert data); RQ2 = from points to reference data
> (segmentation mask, grade and accuracy filters); RQ3 =
> transferability away from the sampled area; RQ4 = TESSERA against
> Sentinel-2.
> In this notebook "RQ1 pilot" is the name of the stored embedding set
> (see `data/README.md`), not a question number. The segmentation mask is RQ2.
>
> **Status (note of 2026-11-13): prototype.** The parameters here
> (maximum radius 20 px, 100 SLIC segments, a sample of 30 presences)
> are not the configuration fixed for RQ2 (5 px, 67 segments; see
> `docs/research_questions.md`). The patch sizes printed below describe
> this prototype, and the COS-polygon candidate is not a condition of
> RQ2.

## 0. Setup and sample

A fixed random sample of 30 presences, to prototype quickly (not the
full 2 465); the final rule, once chosen, is applied to all of them.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
COS = ROOT / "data" / "interim" / "spatial" / "COS2023v1-S2.gpkg"
OUT = ROOT / "data" / "interim" / "spatial" / "qgis" / "segmentation_mask_sample.gpkg"

acacia = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
acacia = gpd.GeoDataFrame(acacia, geometry=gpd.points_from_xy(acacia.longitude, acacia.latitude), crs=4326).to_crs(3763)
sample = acacia.sample(30, random_state=0).reset_index(drop=True)
print(f"{len(acacia):,} presences; sample: {len(sample)}")
sample[["id", "pixel", "geometry"]].to_file(OUT, layer="sample_points", driver="GPKG")

## 1. COS 2023 polygon

No new data or dependency needed: `data/interim/spatial/COS2023v1-S2.gpkg`
already has the "Florestas de acácias" class, already used in
`03_expert_reference/expert_polygons.ipynb` and
`02_citizen_science/acacia_observations.ipynb`. Known limitation
(checked on the full 2 465): only ~18% of presences fall
inside one of these polygons, so this rule alone leaves most
observations without a mask.

In [ ]:
buffer = sample.buffer(200)  # generous margin around the sample so the polygon join doesn't miss edge cases
cos_acacia = gpd.read_file(COS, layer="COS2023v1", where="COS23_n4_C = '5.1.1.7'", bbox=tuple(buffer.total_bounds))
mask_cos = gpd.sjoin(sample[["id", "geometry"]], cos_acacia[["geometry"]], predicate="within", how="left")
mask_cos = mask_cos.dropna(subset=["index_right"]).merge(
    cos_acacia.geometry.rename("mask_geometry"), left_on="index_right", right_index=True)
mask_cos = gpd.GeoDataFrame(mask_cos[["id"]], geometry=mask_cos["mask_geometry"], crs=3763)
print(f"{len(mask_cos)}/{len(sample)} sample points fall inside a COS acacia polygon")
if len(mask_cos):
    mask_cos.to_file(OUT, layer="mask_cos", driver="GPKG")

## 2. Region growing on TESSERA embeddings

**Update (2026-11-03): unblocked.** The dead end above was only the
high-level `GeoTessera` class (`fetch_embedding`,
`fetch_mosaic_for_region`, `sample_embeddings_at_points`), which reads
the `cambridge` NPY dataset only — no Portugal coverage, confirmed
directly against the bucket. The library has a separate, lower-level
class, `geotessera.store.GeoTesseraZarr`, that reads the `dclimate`
dataset (the one that actually covers Portugal) straight from its
Zarr/Icechunk store over HTTPS — this is the same store the deleted
RQ1 pilot already used, just for single pixels
(`sample_points`) instead of a patch (`read_patch`). See
[`notebooks/04_satellite/tessera_embeddings.ipynb`](../04_satellite/tessera_embeddings.ipynb)
for the working access method, confirmed with a real patch (21×21 px,
210×210 m, no NaN) around a presence.

Superpixels (candidate 3) can now also use this patch as its input
raster — still not attempted here.

**Rule (my own, not from a paper — see above):** read a 41×41 px
(410×410 m) patch centred on the point; starting from the centre
pixel, flood-fill to 4-connected neighbours whose embedding has
cosine similarity ≥ a threshold to the centre's, up to 20 px (200 m)
from the centre. Threshold and radius are **not grounded in anything
— guesses, to be calibrated by looking at the result**, not a decided
rule.

In [ ]:
import numpy as np
from geotessera.store import GeoTesseraZarr
from rasterio.features import shapes as raster_shapes
from shapely.geometry import shape as shapely_shape
from shapely.ops import unary_union

store = GeoTesseraZarr(cache_dir=ROOT / "data" / "raw" / "satellite" / "tessera_zarr_cache")
YEAR = 2023
SIZE_PX = 41
SIM_THRESHOLD = 0.90
MAX_RADIUS_PX = 20


def grow_region(patch, threshold=SIM_THRESHOLD, max_radius=MAX_RADIUS_PX):
    h, w, _ = patch.shape
    seed = (h // 2, w // 2)
    seed_emb = patch[seed]
    sim = (patch @ seed_emb) / (np.linalg.norm(patch, axis=-1) * np.linalg.norm(seed_emb))
    mask = np.zeros((h, w), dtype=bool)
    mask[seed] = True
    stack = [seed]
    while stack:
        r, c = stack.pop()
        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
            nr, nc = r + dr, c + dc
            if (0 <= nr < h and 0 <= nc < w and not mask[nr, nc]
                    and max(abs(nr - seed[0]), abs(nc - seed[1])) <= max_radius
                    and sim[nr, nc] >= threshold):
                mask[nr, nc] = True
                stack.append((nr, nc))
    return mask


sample_wgs84 = sample.to_crs(4326)
rows, pixel_counts = [], []
for i, row in sample.iterrows():
    lon, lat = sample_wgs84.geometry.iloc[i].x, sample_wgs84.geometry.iloc[i].y
    patch, transform, crs = store.read_patch(lon=lon, lat=lat, year=YEAR, size_px=SIZE_PX)
    mask = grow_region(patch)
    pixel_counts.append(int(mask.sum()))
    polys = [shapely_shape(geom) for geom, val in raster_shapes(mask.astype("uint8"), mask=mask, transform=transform) if val]
    rows.append({"id": row["id"], "n_pixels": int(mask.sum()), "geometry": unary_union(polys)})

mask_tessera = gpd.GeoDataFrame(rows, crs=crs).to_crs(3763)
mask_tessera.to_file(OUT, layer="mask_tessera", driver="GPKG")
print(f"pixels per mask: min {min(pixel_counts)}, median {int(np.median(pixel_counts))}, max {max(pixel_counts)} "
      f"(1 pixel = seed only; {MAX_RADIUS_PX * 2 + 1}x{MAX_RADIUS_PX * 2 + 1} = hit the window edge)")

**Check one example** before trusting the numbers above: PCA-RGB of
its patch, with the grown region outlined.

In [ ]:
from sklearn.decomposition import PCA

example = int(np.argsort(pixel_counts)[len(pixel_counts) // 2])  # a median-sized one, not a trivial min/max
lon, lat = sample_wgs84.geometry.iloc[example].x, sample_wgs84.geometry.iloc[example].y
patch, transform, crs = store.read_patch(lon=lon, lat=lat, year=YEAR, size_px=SIZE_PX)
mask = grow_region(patch)

rgb = PCA(n_components=3).fit_transform(patch.reshape(-1, patch.shape[-1])).reshape(*patch.shape[:2], 3)
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(rgb)
ax.contour(mask, levels=[0.5], colors="red", linewidths=2)
ax.scatter([SIZE_PX // 2], [SIZE_PX // 2], c="red", marker="x", s=80)
ax.set_title(f"id {sample.loc[example, 'id']}: {pixel_counts[example]} pixels "
             f"(threshold {SIM_THRESHOLD}, max radius {MAX_RADIUS_PX} px)")
plt.show()

## 3. Superpixels (SLIC on TESSERA embeddings)

Rivier et al.'s idea only, as scoped in
[`thesis/notes.md`](../../thesis/notes.md#máscara-de-segmentação-testar-várias-abordagens):
superpixels + majority-vote label propagation, applied to TESSERA
embedding patches, not their full method (they train a segmentation
network from scratch on sub-metre RGB imagery with DAL-HERS
superpixels — see
[`docs/literature.md`](../../docs/literature.md#weak-supervision-from-point-labels)
for the differences). Here SLIC (Achanta et al., 2012, via
`skimage.segmentation.slic`) treats the patch's 128 embedding
dimensions as its channels, instead of RGB, and the superpixel
containing the seed pixel is taken as the mask. Majority-vote
propagation itself is not exercised in this sample — one point per
patch, nothing yet to vote over; only the superpixel geometry is
compared here, same scope as the region-growing candidate above.

**Parameters — not grounded, to calibrate visually (same caveat as
the region-growing candidate above):** `n_segments=100` and
`compactness=0.1` on the 41×41 px patch — not a literature
value. Rivier et al.'s own ablation found fewer, larger superpixels
(80-100 per 512×512 image) work better than many small ones in their
setting, but that is a different scale and imagery (dense RGB point
grids, not sparse citizen-science points on embeddings), so it is
noted as context, not transferred as a parameter.

In [ ]:
from skimage.segmentation import slic, mark_boundaries

N_SEGMENTS = 100  # ~17 px/segment on a 41x41 patch, chosen to land near the region-growing median (15 px) — not grounded in a paper
COMPACTNESS = 0.1  # low: let embedding similarity dominate over spatial compactness; not grounded, to calibrate visually


def superpixel_mask(patch, n_segments=N_SEGMENTS, compactness=COMPACTNESS):
    h, w, _ = patch.shape
    segments = slic(patch, n_segments=n_segments, compactness=compactness, channel_axis=-1, start_label=0)
    seed_label = segments[h // 2, w // 2]
    return segments == seed_label


rows, pixel_counts_sp = [], []
for i, row in sample.iterrows():
    lon, lat = sample_wgs84.geometry.iloc[i].x, sample_wgs84.geometry.iloc[i].y
    patch, transform, crs = store.read_patch(lon=lon, lat=lat, year=YEAR, size_px=SIZE_PX)
    mask = superpixel_mask(patch)
    pixel_counts_sp.append(int(mask.sum()))
    polys = [shapely_shape(geom) for geom, val in raster_shapes(mask.astype("uint8"), mask=mask, transform=transform) if val]
    rows.append({"id": row["id"], "n_pixels": int(mask.sum()), "geometry": unary_union(polys)})

mask_superpixels = gpd.GeoDataFrame(rows, crs=crs).to_crs(3763)
mask_superpixels.to_file(OUT, layer="mask_superpixels", driver="GPKG")
print(f"pixels per mask: min {min(pixel_counts_sp)}, median {int(np.median(pixel_counts_sp))}, max {max(pixel_counts_sp)}")

**Check the same median example as the region-growing candidate**,
for a direct visual comparison — superpixel boundaries in yellow, the
chosen (seed) superpixel outlined in red.

In [ ]:
example_sp = int(np.argsort(pixel_counts_sp)[len(pixel_counts_sp) // 2])
lon, lat = sample_wgs84.geometry.iloc[example_sp].x, sample_wgs84.geometry.iloc[example_sp].y
patch, transform, crs = store.read_patch(lon=lon, lat=lat, year=YEAR, size_px=SIZE_PX)
segments = slic(patch, n_segments=N_SEGMENTS, compactness=COMPACTNESS, channel_axis=-1, start_label=0)
mask = segments == segments[SIZE_PX // 2, SIZE_PX // 2]

rgb = PCA(n_components=3).fit_transform(patch.reshape(-1, patch.shape[-1])).reshape(*patch.shape[:2], 3)
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(mark_boundaries(rgb, segments, color=(1, 1, 0)))
ax.contour(mask, levels=[0.5], colors="red", linewidths=2)
ax.scatter([SIZE_PX // 2], [SIZE_PX // 2], c="red", marker="x", s=80)
ax.set_title(f"id {sample.loc[example_sp, 'id']}: {pixel_counts_sp[example_sp]} pixels "
             f"(SLIC, n_segments {N_SEGMENTS}, compactness {COMPACTNESS})")
plt.show()

## 4. Summary so far — not a decision yet

No rule is chosen here, per the purpose stated at the top of this
notebook. What the sample of 30 shows:

- **COS acacia polygon:** only covers points that happen to fall
  inside a mapped "Florestas de acácias" polygon (~18% on the full
  2 465) — not a per-point grown mask, so its
  pixel-count distribution is not comparable to the other two.
- **Region growing (TESSERA embeddings):** min 3, median 33, max 962
  pixels. The upper tail (962 px ≈ 9.6 ha, hitting the 20 px/200 m
  radius cap) suggests the 0.90 similarity threshold is too permissive
  for some patches — it keeps growing into surrounding vegetation
  rather than stopping at the acacia stand.
- **Superpixels (SLIC on TESSERA embeddings):** min 10, median 22, max
  64 pixels — far more consistent, but only because `n_segments` caps
  the maximum size by construction, not because the segmentation is
  more accurate. Pixel counts alone cannot judge which is "right."

**Not done yet:**
- Visual comparison of all three in QGIS — `mask_cos`, `mask_tessera`
  and `mask_superpixels` are now all in
  `data/interim/spatial/qgis/segmentation_mask_sample.gpkg`, alongside
  `sample_points`.
- Superpixel majority-vote label propagation (Rivier et al.'s actual
  proposal) — not exercised here, since this sample has one point per
  patch.
- Choosing between candidates, and the ablation comparing the chosen
  rule against no mask at all — see `docs/todo.md`.